# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and exports
trades for a training-selected wallet universe.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [9]:
# TO recalculate: 
# remove token_df file
# remove enhanced trades
# remove processed trades
# remove processed markets

# rm /Users/vobornij/projects/polymarket/data/markets_processed/*.parquet
# rm /Users/vobornij/projects/polymarket/data/trades_polygon_enriched/enriched_*
# rm /Users/vobornij/projects/polymarket/data/polygon_trades_processed/*.parquet

In [10]:
%load_ext autoreload
%autoreload 2

import json
import datetime
from pathlib import Path

import pandas as pd

import numpy as np
np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [11]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2025, 1, 1)
END_DATE   = datetime.date(2027, 4, 5)

# ── train / test split ────────────────────────────────────────────────────────
# contracts with resolution before end_train_ts are used for training, and contracts with resolution after end_train_ts are used for testing.
END_TRAIN_TS = datetime.datetime(2026, 6, 1, tzinfo=datetime.timezone.utc)

COPY_WINDOW_SECONDS = 5 * 60  # 5 minutes

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
ENRICHED_TRADES_DIR  = Path("../../data/trades_polygon_enriched")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


In [12]:
from polymarket_analysis.data.data_catalogue import (
    load_markets_processed,
    build_token_lookup_df,
    load_token_lookup_df,
)
token_df = load_token_lookup_df()
print(f"Token lookup rows: {len(token_df):,}")

token_df = token_df[
    ~(token_df['primary_tag'].isin(['Sports', 'Crypto']))
    & token_df['token_winner'].notna()
    ]

# Per-condition last trade timestamp (across all raw trades globally)
condition_max_ts = []
for f in sorted(TRADES_DIR.glob("*.parquet")):
    df = pd.read_parquet(f, columns=["block_timestamp", "condition_id"])
    condition_max_ts.append(df.groupby("condition_id")["block_timestamp"].max())
condition_max_ts = pd.concat(condition_max_ts).groupby(level=0).max().reset_index()
condition_max_ts.columns = ["condition_id", "block_timestamp"]
condition_max_ts["last_condition_trade_ts"] = pd.to_datetime(condition_max_ts["block_timestamp"], unit="s", utc=True)
token_df = token_df.merge(condition_max_ts[["condition_id", "last_condition_trade_ts"]], on="condition_id", how="left")
print(f"Added last_condition_trade_ts for {len(condition_max_ts):,} conditions")

len(token_df)

Token lookup rows: 3,374,530
Added last_condition_trade_ts for 127,888 conditions


419552

## 2 · Process trades — Phase 1: select top-4% wallets per shard

Each shard is processed independently in parallel. Only per-wallet training-period P&L
is returned (no trade rows), so memory usage is minimal. The union of top-4% wallets
across all shards becomes the candidate set for Phase 2.

Selection here is based on **training trade_pnl** (not copyability).


In [13]:
# Preprocess: enrich by copyable quantity if not present

from concurrent.futures import ProcessPoolExecutor, as_completed

from polymarket_analysis.preprocessing.fill_extender import enrich_shard

trade_files = sorted(TRADES_DIR.glob("*.parquet"))

with ProcessPoolExecutor() as executor:
    futures = {executor.submit(enrich_shard, f, ENRICHED_TRADES_DIR, COPY_WINDOW_SECONDS, token_df): f for f in trade_files}

    total = len(futures)

    for i, fut in enumerate(as_completed(futures), start=1):
        f = futures[fut]
        try:
            fut.result()
        except Exception as e:
            print(f"Error processing {f}: {e}")
            continue

        if i % 4 == 0 or i == total:
            print(f"{i}/{total} shards processed")

Enriched file for 0.parquet already exists, skipping...
Enriched file for 1.parquet already exists, skipping...
Enriched file for 2.parquet already exists, skipping...
Enriched file for 3.parquet already exists, skipping...
4/16 shards processed
Enriched file for 4.parquet already exists, skipping...
Enriched file for 5.parquet already exists, skipping...
Enriched file for 6.parquet already exists, skipping...
Enriched file for 7.parquet already exists, skipping...
8/16 shards processed
Enriched file for 8.parquet already exists, skipping...
Enriched file for 9.parquet already exists, skipping...
Enriched file for a.parquet already exists, skipping...
Enriched file for b.parquet already exists, skipping...
12/16 shards processed
Enriched file for c.parquet already exists, skipping...
Enriched file for d.parquet already exists, skipping...
Enriched file for e.parquet already exists, skipping...
Enriched file for f.parquet already exists, skipping...
16/16 shards processed


In [14]:
from polymarket_analysis.preprocessing.shard_processor import select_top_wallets_shard

# ── load token resolution DataFrame ─────────────────────────────────────────
enriched_trade_files = sorted(ENRICHED_TRADES_DIR.glob("*.parquet"))

# ── Phase 1: identify top-4% wallets per shard (parallel) ────────────────────
print("Phase 1 — selecting top-4% wallets per shard (trade_pnl)...")
shard_wallet_pnls: dict[str, float] = {}   # wallet → best shard pnl (for diagnostics)
shard_wallet_thresholds: dict[int, float] = {}   # shard index → top-pct threshold (for diagnostics)
total_raw_rows      = 0
total_in_range_rows = 0
total_candidates    = 0

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(
            select_top_wallets_shard,
            f,
            token_df,
            END_TRAIN_TS,
            top_pct=0.04,
            selection_pnl="trade_pnl",
        ): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        wallet_pnl, stats = fut.result()
        total_raw_rows      += stats["raw_rows"]
        total_in_range_rows += stats["in_range_rows"]
        total_candidates    += stats["candidate_wallets"]
        # union: keep the wallet; if it appears in multiple shards take max pnl
        for w, pnl in wallet_pnl.items():
            if w not in shard_wallet_pnls or pnl > shard_wallet_pnls[w]:
                shard_wallet_pnls[w] = pnl
        shard_wallet_thresholds[i] = stats["threshold"]
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(
                f"  {i}/{len(enriched_trade_files)} shards | "
                f"raw: {total_raw_rows:,} | in-range: {total_in_range_rows:,} | "
                f"wallet union so far: {len(shard_wallet_pnls):,}"
            )

top_wallets: set[str] = set(shard_wallet_pnls.keys())
print(f"\nPhase 1 done. Candidate wallets (union of top-4% per shard): {len(top_wallets):,}")
print(f"Threshold pnls per shard: {shard_wallet_thresholds}")


Phase 1 — selecting top-4% wallets per shard (trade_pnl)...
Processing shard enriched_0.parquet...
Processing shard enriched_1.parquet...
Processing shard enriched_2.parquet...
Processing shard enriched_3.parquet...
Shard top 4.00% threshold: 81.00 USDC
Processing shard enriched_4.parquet...
Shard top 4.00% threshold: 85.17 USDC
Processing shard enriched_5.parquet...
Processing shard enriched_6.parquet...
Shard top 4.00% threshold: 81.82 USDC
Processing shard enriched_7.parquet...
Shard top 4.00% threshold: 117.15 USDC
Shard top 4.00% threshold: 111.67 USDC
Processing shard enriched_8.parquet...
  4/16 shards | raw: 23,137,351 | in-range: 23,137,351 | wallet union so far: 28,931
Processing shard enriched_9.parquet...
Shard top 4.00% threshold: 73.00 USDC
Processing shard enriched_a.parquet...
Shard top 4.00% threshold: 102.98 USDC
Processing shard enriched_b.parquet...
Shard top 4.00% threshold: 85.44 USDC
Shard top 4.00% threshold: 91.27 USDC
Processing shard enriched_c.parquet...
  8

## 3 · Phase 2: enrich, group by tx×wallet×side, filter to top wallets

Each shard is processed in parallel.  Fills are inner-joined with settlement data,
aggregated to one row per ``tx_hash × wallet × side``, and filtered to the wallet set
from Phase 1.  Results are concatenated in memory and re-grouped to merge any
transactions that span shard boundaries.

In [15]:
from polymarket_analysis.preprocessing.shard_processor import enrich_and_group_shard

# ── Phase 2: enrich + group + filter (parallel) ──────────────────────────────
print("Phase 2 — grouping and filtering shards...")
shard_results:     list[pd.DataFrame]    = []
wallet_pnl_total:  dict[str, float]      = {}   # accumulated training trade_pnl per wallet
sample_fills = None

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(
            enrich_and_group_shard,
            f,
            token_df,
            END_TRAIN_TS,
            top_wallets,
            wallet_pnl_metric="trade_pnl",
        ): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        grouped_shard, shard_train_pnl = fut.result()
        if not grouped_shard.empty:
            shard_results.append(grouped_shard)
            if sample_fills is None:
                sample_fills = grouped_shard.head(5).copy()
        for w, pnl in shard_train_pnl.items():
            wallet_pnl_total[w] = wallet_pnl_total.get(w, 0.0) + pnl
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(f"  {i}/{len(enriched_trade_files)} shards | shards with data: {len(shard_results)}")

if not shard_results:
    raise ValueError("No in-range trade rows after token filter")

# ── merge cross-shard partial groups ─────────────────────────────────────────
GROUP_KEYS = ["tx_hash", "wallet", "side", "token_id"]
grouped = pd.concat(shard_results, ignore_index=True)
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",               "first"),
        last_condition_trade_ts = ("last_condition_trade_ts", "max"),
        condition_id     = ("condition_id",     "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_x_qty_sum  = ("price_x_qty_sum",  "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
        copyable_pnl     = ("copyable_pnl",     "sum"),
        copyable_qty     = ("copyable_qty",     "sum"),
        avail_copy_total_vol = ("avail_copy_total_vol", "sum"),
        avail_copy_count  = ("avail_copy_count", "sum"),
    )
    .reset_index()
)
grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]
# grouped.drop(columns=["price_x_qty_sum"], inplace=True)
grouped.sort_values(["wallet", "dt"], inplace=True, ignore_index=True)

print(f"\nPhase 2 done.")
print(f"  Grouped rows (all top wallets, all periods): {len(grouped):,}")
print(f"  Unique wallets in grouped:                   {grouped['wallet'].nunique():,}")
grouped.head(5)


Phase 2 — grouping and filtering shards...


  4/16 shards | shards with data: 4
  8/16 shards | shards with data: 8
  12/16 shards | shards with data: 12
  16/16 shards | shards with data: 16

Phase 2 done.
  Grouped rows (all top wallets, all periods): 47,217,126
  Unique wallets in grouped:                   69,541


,tx_hash,wallet,side,token_id,dt,last_condition_trade_ts,condition_id,outcome,token_winner,final_price,...,price_x_qty_sum,trade_value_usdc,final_value_usdc,num_fills,trade_pnl,copyable_pnl,copyable_qty,avail_copy_total_vol,avail_copy_count,avg_price
0,0x0608e4eeb4f28612dfee57a202a790d2f183d4e251f0...,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,2576518463476533936382780799241198667006764167...,2026-05-18 05:53:53+00:00,2026-05-22 06:11:42+00:00,0x098afead2c41677b0f09ae9d0013ca520eacdb3f0d7c...,Yes,False,0.00,...,700.00,700.00,0.00,1,-700.00,-56.59,404.23,52.55,2.00,0.14
1,0xffb7f006adee358b90369094ef388a7eb54d14d54b3c...,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,2859321553434544736215632162083862677465250387...,2026-05-18 05:54:52+00:00,2026-05-27 03:19:19+00:00,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9...,Yes,False,0.00,...,1316.84,1316.84,0.00,7,-1316.84,-0.61,1.84,0.59,4.00,0.33
2,0x2f6aaa6d6f26ca01bb3326f842fdd7d235b4bc2f8f87...,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,2576518463476533936382780799241198667006764167...,2026-05-19 10:50:26+00:00,2026-05-22 06:11:42+00:00,0x098afead2c41677b0f09ae9d0013ca520eacdb3f0d7c...,Yes,False,0.00,...,63.00,63.00,0.00,3,-63.00,-55.60,441.25,78.05,27.00,0.13
3,0xe02e9a4cae3904e6e902aec3a5fd4369b89586e64f48...,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,SELL,2859321553434544736215632162083862677465250387...,2026-05-23 05:15:15+00:00,2026-05-27 03:19:19+00:00,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9...,Yes,False,0.00,...,3947.01,3947.01,0.00,7,3947.01,1288.37,1305.67,1864.22,23.00,0.99
4,0xe1958851124cdf70de4936b01dc7ec0693e412a1847c...,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,5296515671733987169271418666078595131042183191...,2026-05-24 13:03:24+00:00,2026-05-27 06:09:07+00:00,0x421bc1929df1429cf2cb94f80c1ce6a3ed0d1f0b7a27...,No,True,1.00,...,3036.00,3036.00,4600.00,5,1564.00,157.99,464.68,1397.68,19.00,0.66


## 4 · Phase 3: apply final trade-PnL cut-off and export

Compute the true cross-shard training P&L per wallet. Use the **median** of the
per-shard trade-PnL thresholds from Phase 1 as the export cut-off.


In [16]:
# ── wallet summary (full cross-shard training P&L) ───────────────────────────
grouped_train = grouped[grouped["last_condition_trade_ts"] < END_TRAIN_TS]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets     = ("condition_id",    "nunique"),
        num_trades      = ("num_fills",       "sum"),
        total_cost_usdc = ("trade_value_usdc", "sum"),
        trade_pnl       = ("trade_pnl",       "sum"),
        copyable_pnl    = ("copyable_pnl",    "sum"),
    )
    .sort_values("trade_pnl", ascending=False)
    .reset_index()
)

# ── PnL cut-off: median of the per-shard trade_pnl thresholds from Phase 1 ───
import statistics
pnl_cutoff = statistics.median(shard_wallet_thresholds.values())

print(f"\nUnique wallets in training data: {wallet_summary['wallet'].nunique():,}")
print(
    f"wallet_summary['trade_pnl'] quantiles:\n"
    f"{wallet_summary['trade_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}"
)

final_wallets = set(
    wallet_summary[
        wallet_summary["trade_pnl"] >= pnl_cutoff
    ]["wallet"]
)

print(f"\nFinal selected wallets (trade-PnL filter): {len(final_wallets):,}")
print(
    f"final_wallets['trade_pnl'] quantiles:\n"
    f"{wallet_summary[wallet_summary['wallet'].isin(final_wallets)]['trade_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}"
)

print(f"Unique wallets after Phase 2 union:    {len(wallet_summary):,}")
print(f"Per-shard trade-PnL median cut-off:    {pnl_cutoff:,.2f} USDC")
print(f"Wallets passing trade-PnL cut-off:     {len(final_wallets):,}")
wallet_summary.head(5)



Unique wallets in training data: 69,541
wallet_summary['trade_pnl'] quantiles:
0.01   -18568.49
0.10    -1221.09
0.20     -302.82
0.30      -26.84
0.40       90.30
0.50      147.42
0.60      231.39
0.70      396.58
0.80      745.61
0.90     2051.87
0.99    34760.22
Name: trade_pnl, dtype: float64

Final selected wallets (trade-PnL filter): 42,222
final_wallets['trade_pnl'] quantiles:
0.01      89.68
0.10     119.52
0.20     156.76
0.30     206.37
0.40     279.25
0.50     389.26
0.60     549.27
0.70     866.70
0.80    1554.09
0.90    4178.83
0.99   54637.95
Name: trade_pnl, dtype: float64
Unique wallets after Phase 2 union:    69,541
Per-shard trade-PnL median cut-off:    86.01 USDC
Wallets passing trade-PnL cut-off:     42,222


,wallet,num_markets,num_trades,total_cost_usdc,trade_pnl,copyable_pnl
0,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,42,2385,3878059.88,1304494.87,501531.92
1,0xbaa2bcb5439e985ce4ccf815b4700027d1b92c73,642,61276,21665144.78,1253880.34,362149.62
2,0xc6587b11a2209e46dfe3928b31c5514a8e33b784,251,10232,35646317.62,1194251.52,347041.41
3,0x0a854897a06d4999e5b2dde5693609f1428ffe9d,50,5438,3448764.00,1077716.68,150921.77
4,0xde7be6d489bce070a959e0cb813128ae659b5f4b,158,12786,7091676.85,903779.46,473711.28


## 5 · Market-level volume summary

In [17]:
# join market metadata (question, end_date) for display
markets_meta = load_markets_processed()[["condition_id", "question", "end_date_iso"]].rename(
    columns={"end_date_iso": "end_date"}
)
grouped_meta = grouped.merge(
    markets_meta,
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 114,193


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x6d0e09d0f04572d9b1adad84703458b0297bc5603b69...,US forces enter Iran by April 30?,2026-04-30T00:00:00Z,5655,191550,225411060.20,224390531.19
1,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,"US x Iran ceasefire extended by April 22, 2026?",2026-04-21T00:00:00Z,2294,71009,166903009.77,166573002.78
2,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77...,US x Iran ceasefire by April 7?,2026-04-07T00:00:00Z,4714,149376,147082158.88,152652817.81
3,0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3...,US government shutdown Saturday?,2026-01-31T00:00:00Z,5330,250193,134739788.33,138226116.16
4,0x655e5ca101c466b6293aa15e06173b78b293221803d5...,Will Zelenskyy wear a suit before July?,2025-06-30T00:00:00Z,2343,72706,131027234.03,131793533.68
5,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e...,"US x Iran permanent peace deal by June 15, 2026?",2026-06-15T00:00:00Z,3328,188004,120088564.27,145172271.63
6,0xa93b28a6384aefff4e7d5adb2061c444e4a0e95b8ad1...,"Russia x Ukraine ceasefire by May 31, 2026?",2026-05-31T00:00:00Z,1518,38607,111309780.75,111903683.98
7,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,Khamenei out as Supreme Leader of Iran by Febr...,2026-02-28T00:00:00Z,6276,196782,105600889.33,115513220.88
8,0x7cb525e831729325d651017f81cbcb6f1adde5011c7b...,Netanyahu out by March 31?,2026-03-31T00:00:00Z,2865,110432,95077927.82,96083095.78
9,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,"US strikes Iran by February 28, 2026?",2026-01-31T00:00:00Z,7709,199334,73234078.77,89980333.51


## 6 · Wallet statistics (quantiles)

In [18]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")
wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]  = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"] = wallet_stats["num_markets"].round(1)
wallet_stats["copyable_pnl"]    = wallet_stats["copyable_pnl"].round(2)
wallet_stats["trade_pnl"]    = wallet_stats["trade_pnl"].round(2)
wallet_stats

,wallet_count,num_trades,num_markets,copyable_pnl,trade_pnl,wallet_count_complement
quantile,,,,,,
0.00,70,1.00,1.00,-66240.89,-120498.12,69471
0.01,695,1.00,1.00,-11831.30,-18568.49,68846
0.05,3477,2.00,1.00,-2229.21,-3312.24,66064
0.10,6954,4.00,1.00,-884.24,-1221.09,62587
0.25,17385,15.00,3.00,-136.35,-133.29,52156
0.50,34770,57.00,9.00,0.24,147.42,34771
0.75,52156,213.00,29.00,185.39,527.37,17385
0.90,62587,785.00,87.00,751.90,2051.87,6954
0.95,66064,1903.00,184.00,1811.72,5419.41,3477


## 7 · Full enriched trade table

In [19]:
DISPLAY_COLS = [
    "tx_hash", "wallet", "side",
    "dt", "condition_id", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,tx_hash,wallet,side,dt,condition_id,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,num_fills
0,0x0608e4eeb4f28612dfee57a202a790d2f183d4e251f0...,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,2026-05-18 05:53:53+00:00,0x098afead2c41677b0f09ae9d0013ca520eacdb3f0d7c...,Yes,5000.00,5000.00,0.14,False,0.00,700.00,0.00,-700.00,-56.59,1
1,0xffb7f006adee358b90369094ef388a7eb54d14d54b3c...,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,2026-05-18 05:54:52+00:00,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9...,Yes,4000.00,4000.00,0.33,False,0.00,1316.84,0.00,-1316.84,-0.61,7
2,0x2f6aaa6d6f26ca01bb3326f842fdd7d235b4bc2f8f87...,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,2026-05-19 10:50:26+00:00,0x098afead2c41677b0f09ae9d0013ca520eacdb3f0d7c...,Yes,5500.00,500.00,0.13,False,0.00,63.00,0.00,-63.00,-55.60,3
3,0xe02e9a4cae3904e6e902aec3a5fd4369b89586e64f48...,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,SELL,2026-05-23 05:15:15+00:00,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9...,Yes,3917.69,4000.00,0.99,False,0.00,3947.01,0.00,3947.01,1288.37,7
4,0xe1958851124cdf70de4936b01dc7ec0693e412a1847c...,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,BUY,2026-05-24 13:03:24+00:00,0x421bc1929df1429cf2cb94f80c1ce6a3ed0d1f0b7a27...,No,4600.00,4600.00,0.66,True,1.00,3036.00,4600.00,1564.00,157.99,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47217121,0x070c8061d33e893ea6c921e020c1fc3bd318e437687c...,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 20:24:18+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c799...,Yes,200.00,200.00,0.02,False,0.00,4.00,0.00,-4.00,-4.00,1
47217122,0x852015bc27fa54364ba4bd307b710982f8c475e33ff5...,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 23:00:20+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c799...,Yes,4200.00,4000.00,0.00,False,0.00,12.00,0.00,-12.00,-12.00,1
47217123,0x8815e825e712da69beadc37cf8ddc48f2eb2ce9eaec7...,0xffffffe1e093aacd21e4e281e66d543fb0b23455,SELL,2026-02-28 16:39:01+00:00,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,Yes,530.00,50.00,0.51,True,1.00,25.35,50.00,-24.65,-24.65,2
47217124,0xdfaa1fa624e9dee67ac0af157cdb94bafec047326147...,0xffffffe1e093aacd21e4e281e66d543fb0b23455,SELL,2026-02-28 20:48:45+00:00,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,Yes,0.00,500.00,0.93,True,1.00,463.00,500.00,-37.00,-37.00,1


## 8 · Export: apply trade-PnL cut-off and write partitioned parquet

Filter grouped trades to `final_wallets` (wallets above the median per-shard
trade-PnL threshold), then write one parquet shard per first hex character of
`condition_id` after `0x`.


In [20]:
top_wallets = final_wallets
print(f"Wallets selected for export: {len(top_wallets):,}")

Wallets selected for export: 42,222


In [21]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "token_id", "dt",
    "side", "outcome", "position",
    "total_quantity", "avg_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl",
    "token_winner", "final_price",
    "last_condition_trade_ts",
    "tx_hash", "num_fills",
    "is_train","copyable_qty", "avail_copy_total_vol", "avail_copy_count",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

export_df = grouped[grouped["wallet"].isin(top_wallets)].copy()
export_df["is_train"] = export_df["last_condition_trade_ts"] <= END_TRAIN_TS
export_df = export_df[EXPORT_COLS].reset_index(drop=True)

# Partition by the first hex character of condition_id after the "0x" prefix,
# matching the input shard layout (0.parquet … 9.parquet, a.parquet … f.parquet).
export_df["_shard"] = export_df["condition_id"].str[2]

output_files = []
for shard_key, shard_df in export_df.groupby("_shard", sort=True):
    out_file = OUT_DIR / f"{shard_key}.parquet"
    shard_df.drop(columns=["_shard"]).to_parquet(out_file, index=False)
    output_files.append(out_file)

export_df = export_df.drop(columns=["_shard"])
top_trades_preview   = export_df.head(5).copy()
top_trades_count     = len(export_df)
total_top_train_rows = int(export_df["is_train"].sum())

print(f"Total grouped rows exported:  {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
for f in sorted(output_files):
    print(f"  {f.name}  ({pd.read_parquet(f).shape[0]:,} rows)")
print(f"\nSaved partitioned output → {OUT_DIR}")

Total grouped rows exported:  33,706,813
  training rows: 23,002,255
  test rows:     10,704,558
Output shards:  16
  0.parquet  (1,909,068 rows)
  1.parquet  (1,986,779 rows)
  2.parquet  (2,113,870 rows)
  3.parquet  (2,155,196 rows)
  4.parquet  (2,249,010 rows)
  5.parquet  (2,098,120 rows)
  6.parquet  (2,352,187 rows)
  7.parquet  (2,160,114 rows)
  8.parquet  (2,137,892 rows)
  9.parquet  (1,992,615 rows)
  a.parquet  (2,252,280 rows)
  b.parquet  (2,092,595 rows)
  c.parquet  (2,022,413 rows)
  d.parquet  (2,223,656 rows)
  e.parquet  (1,923,024 rows)
  f.parquet  (2,037,994 rows)

Saved partitioned output → ../../data/polygon_trades_processed


In [22]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview

In [23]:
df = pd.read_parquet(ENRICHED_TRADES_DIR/'enriched_3.parquet')
len(df)

6125865

In [24]:
len(df[(df['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20')])

268526

In [25]:
dfs = []
for f in sorted(ENRICHED_TRADES_DIR.glob("*.parquet")):
    dfp = pd.read_parquet(f)
    dfs.append(dfp[dfp['wallet'] == '0x96489abcb9f583d6835c8ef95ffc923d05a86825'])

df_full = pd.concat(dfs, ignore_index=True)

In [26]:
pd.set_option('display.max_columns', None)

In [27]:
enriched = df_full.copy()
enriched["dt"] = pd.to_datetime(enriched["block_timestamp"], unit="s", utc=True)
enriched['final_price'] = enriched['token_winner'] * 1
enriched["final_value_usdc"] = enriched["quantity"] * enriched["final_price"]
enriched["price_x_qty"] = enriched["price"] * enriched["quantity"]

_GROUP_KEYS = ["tx_hash", "wallet", "side"]
# Group fills → one row per tx_hash × wallet × side
grouped = (
    enriched.groupby(_GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",              "first"),
        condition_id     = ("condition_id",    "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("quantity",         "sum"),
        price_x_qty_sum  = ("price_x_qty",     "sum"),
        trade_value_usdc = ("usdc_amount",      "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("log_index",        "count"),
        copyable_qty    = ("copyable_qty",   "sum"),
        avail_copy_total_vol = ("avail_copy_total_vol", "sum"),
        avail_copy_count  = ("avail_copy_count", "sum"),
    )
    .reset_index()
)

grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]

is_buy = grouped["side"] == "BUY"
grouped["trade_pnl"] = np.where(
    is_buy,
    grouped["final_value_usdc"] - grouped["trade_value_usdc"],
    grouped["trade_value_usdc"] - grouped["final_value_usdc"],
)

In [28]:
grouped.groupby('condition_id').agg(
    total_trades = ('num_fills', 'sum'),
    total_volume = ('trade_value_usdc', 'sum'),
    total_pnl    = ('trade_pnl', 'sum'),
).sort_values('total_pnl', ascending=True).head(10)

,total_trades,total_volume,total_pnl
condition_id,,,
0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,2838,1589594.90,-1589594.90
0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77f981fad98f2bfa0b1bc7,1443,745147.28,-631003.04
0x4b02efe53e631ada84681303fd66d79ad615f3d2b6a28b4633d43d935f89af58,1555,585240.20,-577608.94
0xdc4b9ecbfac2c8e98fe18b35c4578a3c64e34ddfe16b1cf8a98ab3ad9d06c088,962,463369.04,-463369.04
0x15aa3c1259a716915e068a0d63c3885d2301d29e8982cbb1717ecb9b63d02d95,857,448182.82,-448182.82
0x70909f0ba8256a89c301da58812ae47203df54957a07c7f8b10235e877ad63c2,1256,477185.87,-419092.55
0x291f0f6023efa933d36cc80fd55f9d176309a92b61b9567fd4f044a8b21873fb,769,409423.70,-409423.70
0x797d586ad45522306490b0cc9b2f21bdf957f3843476fae99f3bcc2cec83b74b,2524,336785.66,-334649.02
0x3fc804e4d893f86d49229eb42492cabf6150f5bcbf36c1b66d1e8f22bc15ec6d,414,285557.54,-285557.54


In [29]:
grouped.groupby(grouped['dt'].dt.date).agg(
    total_trades = ('num_fills', 'sum'),
    total_volume = ('trade_value_usdc', 'sum')
).sort_index(ascending=False).head(10)

,total_trades,total_volume
dt,,
2026-07-15,20,4366.79
2026-07-14,2,5791.03
2026-07-13,3,350.64
2026-07-11,3,418.66
2026-07-08,6,702.00
2026-07-07,1,0.01
2026-07-06,9,6.00
2026-07-05,30,17.70
2026-07-04,12,253.68


In [30]:
df = pd.read_parquet(OUT_DIR/'0.parquet')

In [32]:
df = df[(df['wallet'] == '0x85d22a973e817f87afe0a7fd346a39823cce7479')
    & (df['condition_id'] == '0x000f8958f3be79021a7fa7c4513f450b46da408aba3e0779596bc20cbb64a195')]

In [1]:
import sys

# These are the usual ipython objects, including this one you are creating
ipython_vars = ['In', 'Out', 'exit', 'quit', 'get_ipython', 'ipython_vars']

# Get a sorted list of the objects and their sizes
sorted([(x, sys.getsizeof(globals().get(x))) for x in dir() if not x.startswith('_') and x not in sys.modules and x not in ipython_vars], key=lambda x: x[1], reverse=True)


[('open', 160)]